## Phase 3 — Lecture par saison & Phase 4 — Score de fiabilité

Reprend les conventions déjà validées en Phase 1 (`notebook/phase1_causes.ipynb`) et Phase 2
(`notebook/phase2_exploration.ipynb`) : familles de causes (Réseau / Voyageurs / Externe),
`cause_fiable`, classes de durée, années d'octobre à septembre, liaisons internationales.

**Phase 3** : les contributions des trois familles de causes varient-elles selon la saison ?

**Phase 4** : score de fiabilité V1 (cadrage, section 3) — part de trains en retard (seuil de la
classe de durée), taux d'annulation, retard moyen tous trains, rang par classe de durée, sur les
12 derniers mois.

Ce notebook sert de brouillon avant de porter le code définitif dans `railinsight/analysis/causes.py`
et `railinsight/analysis/reliability.py`.


### 1. Préparation : imports et chargement des données

Mêmes imports que les notebooks précédents, plus `preprocess_dates` pour obtenir la saison
de chaque observation.


In [1]:
import pandas as pd

from railinsight.data.loader import REGULARITE, load
from railinsight.data.preprocessing import preprocess_dates
from railinsight.data.quality import normaliser_noms

liaison = ["gare_depart", "gare_arrivee"]


In [2]:
# Chargement et nettoyage des noms de gare (identique Phase 1 / Phase 2)
df = load(REGULARITE)

for col in liaison:
    df[col] = normaliser_noms(df[col])

print(df.shape)


(12907, 26)


### 2. Récap Phase 1 & 2 : variables réutilisées

On redéfinit exactement les mêmes éléments que dans les notebooks précédents, pour ne pas
dépendre d'un ordre d'exécution :
- les **3 familles de causes** (Réseau / Voyageurs / Externe) et `cause_fiable` (Phase 1),
- la **classe de durée** par liaison et les **années d'octobre à septembre** (Phase 2),
- la liste des **gares étrangères** (Phase 1 & 2).


In [3]:
# Familles de causes (Phase 1 : SNCF Réseau / SNCF Voyageurs / Externe, cf. cadrage)
familles = {
    "reseau": ["prct_cause_infra", "prct_cause_gestion_trafic"],
    "voyageurs": [
        "prct_cause_materiel_roulant",
        "prct_cause_gestion_gare",
        "prct_cause_prise_en_charge_voyageurs",
    ],
    "externe": ["prct_cause_externe"],
}
toutes_causes = familles["reseau"] + familles["voyageurs"] + familles["externe"]

# cause_fiable : même règle qu'en Phase 1 (seuil de 5 points retenu après étude de la distribution)
somme_causes = df[toutes_causes].sum(axis=1)
a_des_retards = df["nb_train_retard_arrivee"] > 0
df["cause_fiable"] = ~(a_des_retards & ((somme_causes - 100).abs() > 5))

# Nombre de trains en retard par famille (pondéré par le nombre de trains en retard)
for famille, colonnes in familles.items():
    df[f"nb_retard_{famille}"] = df["nb_train_retard_arrivee"] * df[colonnes].sum(axis=1) / 100

print(df["cause_fiable"].value_counts())


cause_fiable
True     12824
False       83
Name: count, dtype: int64


In [4]:
# Classe de durée par liaison (médiane, Phase 2) : seuils 90 / 180 minutes
duree_liaison = (
    df.groupby(liaison)["duree_moyenne"].median().rename("duree_liaison").reset_index()
)
duree_liaison["classe_duree"] = pd.cut(
    duree_liaison["duree_liaison"],
    bins=[0, 90, 180, float("inf")],
    labels=["< 1h30", "1h30-3h", "> 3h"],
)
df = df.merge(duree_liaison, on=liaison)

# Années d'octobre à septembre (Phase 2) : octobre-décembre comptent pour l'année suivante
mois = df["date"].str[5:7].astype(int)
df["annee"] = df["date"].str[:4].astype(int) + (mois >= 10).astype(int)

# Gares étrangères (Phase 1 & 2)
gares_etrangeres = [
    "BARCELONA", "MADRID", "FRANCFORT", "GENEVE", "LAUSANNE", "STUTTGART", "ZURICH", "ITALIE",
]
df["internationale"] = df["gare_depart"].isin(gares_etrangeres) | df["gare_arrivee"].isin(
    gares_etrangeres
)

print(df[["classe_duree", "annee", "internationale"]].describe(include="all"))


       classe_duree         annee internationale
count         12907  12907.000000          12907
unique            3           NaN              2
top         1h30-3h           NaN          False
freq           5574           NaN          11357
mean            NaN   2022.081816            NaN
std             NaN      2.541696            NaN
min             NaN   2018.000000            NaN
25%             NaN   2020.000000            NaN
50%             NaN   2022.000000            NaN
75%             NaN   2024.000000            NaN
max             NaN   2026.000000            NaN


## Phase 3 — Lecture par saison

Question du cadrage : *les causes de retard varient-elles selon la saison (travaux l'été,
intempéries l'hiver, affluence lors des vacances scolaires) ?*

Décision du cadrage : la saisonnalité est un **axe de lecture des causes**, pas une catégorie de
cause à part. On réutilise la fonction `preprocess_dates` (saison météorologique : hiver,
printemps, été, automne) et on calcule, **toutes liaisons confondues**, la contribution de
chaque famille de causes par saison — jamais de simple moyenne de pourcentages, toujours
pondéré par le nombre de trains (cf. décisions validées, section 2 du cadrage).


In [5]:
# Table de dates (saison météorologique) via le preprocessing commun
dates = preprocess_dates(df, colonne_date="date")
df = df.merge(dates[["date", "saison"]], on="date", how="left")

print(dates[["date", "saison"]].head(3))
print(df["saison"].value_counts())


      date     saison
0  2018-01      hiver
1  2018-02      hiver
2  2018-03  printemps
saison
été          3337
printemps    3307
hiver        3185
automne      3078
Name: count, dtype: int64


In [6]:
# Contribution de chaque famille par saison, toutes liaisons confondues
# (observations fiables uniquement, même règle qu'en Phase 1 et Phase 2 étape 5B)
df_ok = df[df["cause_fiable"]]

par_saison = df_ok.groupby("saison", observed=True).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_retard_reseau=("nb_retard_reseau", "sum"),
    nb_retard_voyageurs=("nb_retard_voyageurs", "sum"),
    nb_retard_externe=("nb_retard_externe", "sum"),
)

for famille in familles:
    par_saison[f"contrib_{famille}"] = (
        par_saison[f"nb_retard_{famille}"] / par_saison["nb_train_prevu"]
    )

colonnes_contrib = [f"contrib_{famille}" for famille in familles]
ordre_saisons = ["hiver", "printemps", "été", "automne"]
par_saison = par_saison.reindex(ordre_saisons)

print(par_saison[colonnes_contrib].round(3))


           contrib_reseau  contrib_voyageurs  contrib_externe
saison                                                       
hiver               0.059              0.046            0.033
printemps           0.051              0.042            0.027
été                 0.062              0.066            0.036
automne             0.064              0.052            0.030


In [7]:
# Cause dominante par saison (en points de taux de retard)
print(par_saison[colonnes_contrib].idxmax(axis=1))


saison
hiver           contrib_reseau
printemps       contrib_reseau
été          contrib_voyageurs
automne         contrib_reseau
dtype: str


### Bilan Phase 3 : la saisonnalité des causes

- Le **Réseau** domine en hiver, au printemps et en automne (5,9 à 6,4 points de taux de retard).
- L'**été** est la seule saison où les **Voyageurs** passent devant (6,6 points), avec aussi le
  pic du Réseau au global le plus proche — cohérent avec l'affluence voyageurs en période de
  vacances et les travaux d'été évoqués dans le brief.
- L'**Externe** reste la famille la plus stable et la plus faible (2,7 à 3,6 points), sans pic
  saisonnier marqué (les grèves, incluses ici, ne sont pas isolables — cf. cadrage).
- Le **printemps** a les contributions les plus basses des trois familles : saison la plus calme
  dans l'ensemble.

**Limites** : mêmes que la Phase 1 (une seule cause retenue par train, 83 observations
`cause_fiable=False` exclues) ; lecture toutes liaisons confondues, sans distinction de classe de
durée ni d'année (une saison regroupe jusqu'à 8-9 années de données, 2018 à 2026).

**Piste de robustesse à tester** : refaire le calcul par saison **et par année** pour vérifier que
le classement (Réseau > Voyageurs > Externe, sauf l'été) est stable d'une année sur l'autre et pas
tiré par une seule année atypique (2018 incomplète, 2020-2021 Covid).


In [8]:
# Test de robustesse : cause dominante par saison et par année civile
df_ok["annee_civile"] = df_ok["date"].str[:4].astype(int)

par_saison_annee = df_ok.groupby(["annee_civile", "saison"], observed=True).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_retard_reseau=("nb_retard_reseau", "sum"),
    nb_retard_voyageurs=("nb_retard_voyageurs", "sum"),
    nb_retard_externe=("nb_retard_externe", "sum"),
)
for famille in familles:
    par_saison_annee[f"contrib_{famille}"] = (
        par_saison_annee[f"nb_retard_{famille}"] / par_saison_annee["nb_train_prevu"]
    )

dominante = par_saison_annee[colonnes_contrib].idxmax(axis=1)
print(dominante.groupby("saison", observed=True).value_counts(normalize=True).round(2))


saison                      
automne    contrib_reseau       0.78
           contrib_voyageurs    0.22
hiver      contrib_reseau       0.78
           contrib_voyageurs    0.22
printemps  contrib_reseau       0.67
           contrib_voyageurs    0.33
été        contrib_reseau       0.67
           contrib_voyageurs    0.33
Name: proportion, dtype: float64


**Robustesse, point d'attention** : en regardant année par année, le **Réseau reste la cause
dominante la majorité des années pour chaque saison**, y compris l'été (67 % des étés), alors que
le calcul agrégé toutes années confondues faisait ressortir les **Voyageurs** en été. L'Externe
n'est jamais dominant, ni en agrégé ni par année.

Explication probable : une ou deux années atypiques (fort pic Voyageurs, par ex. 2026 d'après la
Phase 2) peuvent faire basculer la moyenne pondérée sans être majoritaires en nombre d'années.
**Décision retenue pour la suite** : afficher la contribution agrégée des trois familles par
saison (ce que montre le graphique), et signaler ce point dans la datasheet plutôt que de changer
la méthode — cohérent avec l'esprit « contributions pondérées, pas de moyenne de taux » du
cadrage.


### Fonctions Phase 3 (à porter dans `railinsight/analysis/causes.py`)

On encapsule les étapes validées ci-dessus :
- `marquer_cause_fiable` : repère les observations à exclure (somme des causes trop loin de 100).
- `ajouter_nb_retard_par_famille` : calcule le nombre de trains en retard par famille.
- `contributions_par_saison` : agrège par saison et calcule les 3 contributions pondérées.


In [9]:
FAMILLES_CAUSES = {
    "reseau": ["prct_cause_infra", "prct_cause_gestion_trafic"],
    "voyageurs": [
        "prct_cause_materiel_roulant",
        "prct_cause_gestion_gare",
        "prct_cause_prise_en_charge_voyageurs",
    ],
    "externe": ["prct_cause_externe"],
}


def marquer_cause_fiable(
    data: pd.DataFrame, familles: dict[str, list[str]] = FAMILLES_CAUSES, tolerance: float = 5.0
) -> pd.Series:
    """Repère les observations dont la répartition des causes est exploitable.

    Une observation est jugée non fiable si elle a des trains en retard à l'arrivée mais que
    la somme des pourcentages de causes s'écarte de plus de `tolerance` points de 100 (Phase 1 :
    83 observations sur 12 907, concentrées sur les liaisons internationales en 2018).

    Args:
        data: Jeu de régularité, avec les colonnes `prct_cause_*` et `nb_train_retard_arrivee`.
        familles: Groupes de colonnes de causes à sommer.
        tolerance: Écart maximal toléré à 100, en points.

    Returns:
        Série booléenne, `True` si l'observation est exploitable.
    """
    toutes_causes = [colonne for colonnes in familles.values() for colonne in colonnes]
    somme_causes = data[toutes_causes].sum(axis=1)
    a_des_retards = data["nb_train_retard_arrivee"] > 0
    return ~(a_des_retards & ((somme_causes - 100).abs() > tolerance))


def ajouter_nb_retard_par_famille(
    data: pd.DataFrame, familles: dict[str, list[str]] = FAMILLES_CAUSES
) -> pd.DataFrame:
    """Ajoute une colonne `nb_retard_<famille>` par groupe de causes.

    Args:
        data: Jeu de régularité, avec les colonnes `prct_cause_*` et `nb_train_retard_arrivee`.
        familles: Groupes de colonnes de causes à sommer.

    Returns:
        Copie de `data` avec une colonne supplémentaire par famille.
    """
    data = data.copy()
    for famille, colonnes in familles.items():
        data[f"nb_retard_{famille}"] = (
            data["nb_train_retard_arrivee"] * data[colonnes].sum(axis=1) / 100
        )
    return data


def contributions_par_saison(
    data: pd.DataFrame, familles: dict[str, list[str]] = FAMILLES_CAUSES
) -> pd.DataFrame:
    """Contribution de chaque famille de causes au taux de retard, par saison.

    Ne garde que les observations `cause_fiable`. La contribution est pondérée par le nombre de
    trains prévus (jamais une moyenne de taux, cf. cadrage).

    Args:
        data: Jeu de régularité prétraité, avec les colonnes `saison`, `nb_train_prevu` et
            `nb_retard_<famille>` (voir `ajouter_nb_retard_par_famille`).
        familles: Groupes de causes à agréger.

    Returns:
        Un DataFrame indexé par saison, avec une colonne `contrib_<famille>` par famille.
    """
    requis = {"saison", "nb_train_prevu", "cause_fiable"}
    if not requis.issubset(data.columns):
        raise ValueError(f"Colonnes manquantes : {requis.difference(data.columns)}")

    data_ok = data[data["cause_fiable"]]
    colonnes_nb_retard = [f"nb_retard_{famille}" for famille in familles]

    agg = data_ok.groupby("saison", observed=True).agg(
        nb_train_prevu=("nb_train_prevu", "sum"),
        **{colonne: (colonne, "sum") for colonne in colonnes_nb_retard},
    )
    for famille in familles:
        agg[f"contrib_{famille}"] = agg[f"nb_retard_{famille}"] / agg["nb_train_prevu"]

    return agg


### Test et validation des fonctions Phase 3

On reconstruit tout depuis les données brutes avec les fonctions, et on compare au résultat
`par_saison` calculé "à la main" plus haut : les deux doivent être identiques.


In [10]:
df_test = load(REGULARITE)
for col in liaison:
    df_test[col] = normaliser_noms(df_test[col])

df_test["cause_fiable"] = marquer_cause_fiable(df_test)
df_test = ajouter_nb_retard_par_famille(df_test)

dates_test = preprocess_dates(df_test, colonne_date="date")
df_test = df_test.merge(dates_test[["date", "saison"]], on="date", how="left")

par_saison_fn = contributions_par_saison(df_test).reindex(ordre_saisons)

pd.testing.assert_frame_equal(
    par_saison_fn[colonnes_contrib], par_saison[colonnes_contrib], check_exact=False
)
print("OK : les fonctions reproduisent exactement le calcul manuel de la Phase 3.")
par_saison_fn[colonnes_contrib].round(3)


OK : les fonctions reproduisent exactement le calcul manuel de la Phase 3.


,contrib_reseau,contrib_voyageurs,contrib_externe
saison,,,
hiver,0.059,0.046,0.033
printemps,0.051,0.042,0.027
été,0.062,0.066,0.036
automne,0.064,0.052,0.030


## Phase 4 — Score de fiabilité

Rappel du cadrage (section 3) :

| Élément | Décision |
| --- | --- |
| Composantes | Part de trains en retard (seuil de la classe de durée), taux d'annulation, retard moyen de tous les trains (`retard_moyen_tous_trains_arrivee`) |
| Poids | Égaux : un tiers chacune |
| Normalisation | Rang de chaque composante à l'intérieur de la classe de durée |
| Période | Les 12 derniers mois |
| Causes | Toutes causes confondues (score "vécu"), part d'origine interne affichée à côté sans entrer dans le calcul |

**Point d'attention (à documenter dans la datasheet)** : le cadrage prévoit un seuil de retard
différent par classe de durée (5 min pour `< 1h30`, 10 min pour `1h30-3h`, 15 min pour `> 3h`).
Le dataset SNCF ne fournit que `nb_train_retard_arrivee` (seuil standard, non détaillé par classe)
et des paliers fixes (`nb_train_retard_sup_15`, `_sup_30`, `_sup_60`). **On utilise donc
`nb_train_retard_arrivee` comme proxy de la composante "part de trains en retard"**, quelle que
soit la classe de durée — à signaler comme une limite, et question ouverte pour le groupe.


In [11]:
# Les 12 derniers mois disponibles
derniers_mois = sorted(df["date"].unique())[-12:]
print(derniers_mois[0], "→", derniers_mois[-1], f"({len(derniers_mois)} mois)")

recent = df[df["date"].isin(derniers_mois)].copy()

# Trains circulés = prévus - annulés (proposition validée en Phase 2, étape 2 : un train annulé
# ne peut pas être en retard, et les annulations sont déjà comptées à part dans le score).
recent["nb_train_circule"] = recent["nb_train_prevu"] - recent["nb_annulation"]


2025-10 → 2026-09 (12 mois)


In [12]:
# Agrégation par liaison sur les 12 derniers mois (pondéré par le nombre de trains)
par_liaison_12m = recent.groupby(liaison).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_train_circule=("nb_train_circule", "sum"),
    nb_annulation=("nb_annulation", "sum"),
    nb_train_retard_arrivee=("nb_train_retard_arrivee", "sum"),
    classe_duree=("classe_duree", "first"),
    internationale=("internationale", "first"),
)

# Moyenne pondérée du retard moyen tous trains, pondérée par les trains prévus du mois
recent["retard_pondere"] = (
    recent["retard_moyen_tous_trains_arrivee"] * recent["nb_train_prevu"]
)
retard_moyen_12m = (
    recent.groupby(liaison)["retard_pondere"].sum()
    / recent.groupby(liaison)["nb_train_prevu"].sum()
)
par_liaison_12m["retard_moyen_tous_trains"] = retard_moyen_12m

print(par_liaison_12m.shape)
par_liaison_12m.head(3)


(121, 7)


,,nb_train_prevu,nb_train_circule,nb_annulation,nb_train_retard_arrivee,classe_duree,internationale,retard_moyen_tous_trains
gare_depart,gare_arrivee,,,,,,,
AIX EN PROVENCE,PARIS LYON,5200,5151,49,895,> 3h,False,10.580507
ANGERS ST LAUD,PARIS MONTPARNASSE,6569,6474,95,1166,1h30-3h,False,7.572050
ANGOULEME,PARIS MONTPARNASSE,3542,3472,70,681,1h30-3h,False,8.007915


In [13]:
# Les 3 composantes brutes (plus la valeur est haute, moins la liaison est fiable)
par_liaison_12m["part_retard"] = (
    par_liaison_12m["nb_train_retard_arrivee"] / par_liaison_12m["nb_train_circule"]
)
par_liaison_12m["taux_annulation"] = (
    par_liaison_12m["nb_annulation"] / par_liaison_12m["nb_train_prevu"]
)
# retard_moyen_tous_trains déjà calculé

composantes = ["part_retard", "taux_annulation", "retard_moyen_tous_trains"]
print(par_liaison_12m.groupby("classe_duree", observed=True)[composantes].describe().round(3))


             part_retard                                                   \
                   count   mean    std    min    25%    50%    75%    max   
classe_duree                                                                
< 1h30              20.0  0.184  0.047  0.105  0.150  0.183  0.202  0.280   
1h30-3h             52.0  0.180  0.052  0.101  0.143  0.173  0.195  0.339   
> 3h                49.0  0.200  0.080  0.083  0.156  0.189  0.235  0.584   

             taux_annulation         ...                \
                       count   mean  ...    75%    max   
classe_duree                         ...                 
< 1h30                  20.0  0.015  ...  0.023  0.035   
1h30-3h                 52.0  0.011  ...  0.013  0.040   
> 3h                    49.0  0.014  ...  0.018  0.060   

             retard_moyen_tous_trains                                       \
                                count    mean    std    min    25%     50%   
classe_duree                    

In [14]:
# Rang de chaque composante à l'intérieur de la classe de durée (1 = la plus fiable)
# pct=True : rang normalisé entre 0 et 1, comparable entre classes de taille différente
rangs = par_liaison_12m.groupby("classe_duree", observed=True)[composantes].rank(pct=True)
rangs.columns = [f"rang_{c}" for c in composantes]
par_liaison_12m = par_liaison_12m.join(rangs)

# Score de fiabilité : moyenne des 3 rangs, poids égaux. Score bas = liaison fiable.
par_liaison_12m["score_fiabilite"] = par_liaison_12m[rangs.columns].mean(axis=1)

colonnes_vue = ["classe_duree", *composantes, "score_fiabilite"]
par_liaison_12m[colonnes_vue].sort_values("score_fiabilite", ascending=False).head(10).round(3)


classe_duree  part_retard  \
gare_depart          gare_arrivee                                     
DOUAI                PARIS NORD                 < 1h30        0.280   
PARIS EST            STUTTGART                    > 3h        0.584   
                     FRANCFORT                    > 3h        0.334   
ARRAS                PARIS NORD                 < 1h30        0.276   
FRANCFORT            PARIS EST                    > 3h        0.251   
MARSEILLE ST CHARLES TOURCOING                    > 3h        0.233   
MARNE LA VALLEE      MARSEILLE ST CHARLES         > 3h        0.258   
NICE VILLE           PARIS LYON                   > 3h        0.241   
PARIS NORD           DOUAI                      < 1h30        0.236   
STRASBOURG           PARIS EST                 1h30-3h        0.226   

                                           taux_annulation  \
gare_depart          gare_arrivee                            
DOUAI                PARIS NORD                      0.035   
PARIS EST            STUTTGART                       0.032   
                     FRANCFORT                       0.030   
ARRAS                PARIS NORD                      0.031   
FRANCFORT            PARIS EST                       0.030   
MARSEILLE ST CHARLES TOURCOING                       0.026   
MARNE LA VALLEE      MARSEILLE ST CHARLES            0.015   
NICE VILLE           PARIS LYON                      0.016   
PARIS NORD           DOUAI                           0.022   
STRASBOURG           PARIS EST                       0.013   

                                           retard_moyen_tous_trains  \
gare_depart          gare_arrivee                                     
DOUAI                PARIS NORD                               7.156   
PARIS EST            STUTTGART                               24.253   
                     FRANCFORT                               17.016   
ARRAS                PARIS NORD                               6.861   
FRANCFORT            PARIS EST                               14.804   
MARSEILLE ST CHARLES TOURCOING                               14.808   
MARNE LA VALLEE      MARSEILLE ST CHARLES                    15.173   
NICE VILLE           PARIS LYON                              14.997   
PARIS NORD           DOUAI                                    6.552   
STRASBOURG           PARIS EST                                9.308   

                                           score_fiabilite  
gare_depart          gare_arrivee                           
DOUAI                PARIS NORD                      0.983  
PARIS EST            STUTTGART                       0.980  
                     FRANCFORT                       0.946  
ARRAS                PARIS NORD                      0.933  
FRANCFORT            PARIS EST                       0.884  
MARSEILLE ST CHARLES TOURCOING                       0.837  
MARNE LA VALLEE      MARSEILLE ST CHARLES            0.837  
NICE VILLE           PARIS LYON                      0.810  
PARIS NORD           DOUAI                           0.800  
STRASBOURG           PARIS EST                       0.788

In [15]:
# Part d'origine interne (Réseau + Voyageurs) sur les 12 derniers mois : affichée à côté du
# score, sans entrer dans son calcul (cadrage, section 3). Toutes causes confondues pour le
# score lui-même (le score est "vécu" : un retard est un retard).
recent_ok = recent[recent["cause_fiable"]]
causes_12m = recent_ok.groupby(liaison).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_retard_reseau=("nb_retard_reseau", "sum"),
    nb_retard_voyageurs=("nb_retard_voyageurs", "sum"),
    nb_retard_externe=("nb_retard_externe", "sum"),
)
part_interne = (
    causes_12m["nb_retard_reseau"] + causes_12m["nb_retard_voyageurs"]
) / (
    causes_12m["nb_retard_reseau"]
    + causes_12m["nb_retard_voyageurs"]
    + causes_12m["nb_retard_externe"]
)
par_liaison_12m["part_origine_interne"] = part_interne

print(par_liaison_12m["part_origine_interne"].describe().round(3))


count    121.000
mean       0.721
std        0.097
min        0.189
25%        0.667
50%        0.730
75%        0.790
max        0.891
Name: part_origine_interne, dtype: float64


### Bilan Phase 4 : le score de fiabilité V1

- **121 liaisons** sur les 12 derniers mois disponibles (oct. 2025 → sept. 2026), la même période
  de suivi dégradée identifiée en Phase 2.
- Le **score** (moyenne des rangs en pourcentile des 3 composantes, par classe de durée) va de 0
  (plus fiable de sa classe) à 1 (moins fiable).
- Les liaisons les moins fiables recoupent largement celles déjà repérées en Phase 2 comme
  "durablement en difficulté" ou en forte dégradation : Paris Est ↔ Stuttgart/Francfort (données
  internationales suspectes, cf. Phase 2 étape 3), Douai/Arras → Paris Nord.
- La **part d'origine interne** (Réseau + Voyageurs) est en médiane de 73 %, cohérente avec la
  Phase 1 (Réseau et Voyageurs dominants pour 98 % des liaisons).

**Limites à documenter dans la datasheet**
- Le seuil de retard par classe de durée (5/10/15 min) n'est pas disponible dans les données : on
  utilise `nb_train_retard_arrivee` (seuil standard SNCF) comme proxy pour toutes les classes.
  **Question ouverte pour le groupe.**
- Les liaisons internationales ne sont pas exclues ici (contrairement à la proposition validée en
  Phase 2 pour la sélection des liaisons en difficulté) : leurs scores sont donc à lire avec la
  même prudence que leurs taux de retard (données suspectes, cf. Phase 2 étape 3).
- Le score n'est pas tracé dans le temps (cadrage : il est relatif à la période des 12 derniers
  mois, pas comparable d'une période à l'autre).


### Fonctions Phase 4 (à porter dans `railinsight/analysis/reliability.py`)

- `agreger_derniers_mois` : limite le jeu de données aux N derniers mois disponibles.
- `composantes_fiabilite` : calcule `part_retard`, `taux_annulation`, `retard_moyen_tous_trains`
  par liaison.
- `score_fiabilite` : ajoute les rangs par classe de durée et le score moyen (poids égaux).
- `part_origine_interne` : part Réseau + Voyageurs par liaison, à afficher à côté du score.


In [16]:
COMPOSANTES_FIABILITE = ["part_retard", "taux_annulation", "retard_moyen_tous_trains"]


def agreger_derniers_mois(data: pd.DataFrame, n_mois: int = 12) -> pd.DataFrame:
    """Restreint le jeu de données aux `n_mois` mois les plus récents.

    Args:
        data: Jeu de régularité, avec une colonne `date` au format ``AAAA-MM``.
        n_mois: Nombre de mois à conserver.

    Returns:
        Les lignes dont la date fait partie des `n_mois` derniers mois disponibles.
    """
    derniers = sorted(data["date"].unique())[-n_mois:]
    return data[data["date"].isin(derniers)].copy()


def composantes_fiabilite(
    data: pd.DataFrame, liaison_cols: list[str] = liaison
) -> pd.DataFrame:
    """Calcule les 3 composantes brutes du score de fiabilité, par liaison.

    Composantes (cadrage, section 3) : part de trains en retard (sur les trains circulés),
    taux d'annulation, et retard moyen de tous les trains à l'arrivée.

    Args:
        data: Jeu de régularité (idéalement restreint aux 12 derniers mois), avec les colonnes
            `nb_train_prevu`, `nb_annulation`, `nb_train_retard_arrivee`,
            `retard_moyen_tous_trains_arrivee` et `classe_duree`.
        liaison_cols: Colonnes identifiant une liaison.

    Returns:
        Un DataFrame indexé par liaison avec les colonnes de `COMPOSANTES_FIABILITE` et
        `classe_duree`.
    """
    data = data.copy()
    data["nb_train_circule"] = data["nb_train_prevu"] - data["nb_annulation"]
    data["retard_pondere"] = data["retard_moyen_tous_trains_arrivee"] * data["nb_train_prevu"]

    agg = data.groupby(liaison_cols).agg(
        nb_train_prevu=("nb_train_prevu", "sum"),
        nb_train_circule=("nb_train_circule", "sum"),
        nb_annulation=("nb_annulation", "sum"),
        nb_train_retard_arrivee=("nb_train_retard_arrivee", "sum"),
        retard_pondere=("retard_pondere", "sum"),
        classe_duree=("classe_duree", "first"),
    )

    agg["part_retard"] = agg["nb_train_retard_arrivee"] / agg["nb_train_circule"]
    agg["taux_annulation"] = agg["nb_annulation"] / agg["nb_train_prevu"]
    agg["retard_moyen_tous_trains"] = agg["retard_pondere"] / agg["nb_train_prevu"]

    return agg[["classe_duree", *COMPOSANTES_FIABILITE]]


def score_fiabilite(
    data: pd.DataFrame, composantes: list[str] = COMPOSANTES_FIABILITE
) -> pd.DataFrame:
    """Ajoute le rang par classe de durée et le score de fiabilité (poids égaux).

    Le score est la moyenne des rangs en pourcentile des composantes, calculés à l'intérieur de
    la classe de durée de chaque liaison. Un score proche de 0 indique une liaison fiable par
    rapport aux autres liaisons de durée comparable ; proche de 1, une liaison peu fiable.

    Args:
        data: Résultat de `composantes_fiabilite` (colonnes `classe_duree` + `composantes`).
        composantes: Colonnes à utiliser pour le score.

    Returns:
        Copie de `data` avec une colonne `rang_<composante>` par composante et `score_fiabilite`.
    """
    if "classe_duree" not in data.columns:
        raise ValueError("La colonne 'classe_duree' est requise pour normaliser par classe.")

    data = data.copy()
    rangs = data.groupby("classe_duree", observed=True)[composantes].rank(pct=True)
    rangs.columns = [f"rang_{c}" for c in composantes]
    data = data.join(rangs)
    data["score_fiabilite"] = data[rangs.columns].mean(axis=1)
    return data


def part_origine_interne(
    data: pd.DataFrame, familles: dict[str, list[str]] = FAMILLES_CAUSES, liaison_cols: list[str] = liaison
) -> pd.Series:
    """Part des retards d'origine interne (Réseau + Voyageurs), par liaison.

    Affichée à côté du score de fiabilité, sans entrer dans son calcul (cadrage, section 3).
    Ne garde que les observations `cause_fiable`.

    Args:
        data: Jeu de régularité avec `cause_fiable` et `nb_retard_<famille>`
            (voir `marquer_cause_fiable` et `ajouter_nb_retard_par_famille`).
        familles: Groupes de causes ; les clés "reseau" et "voyageurs" sont utilisées comme
            origine interne, les autres clés comme origine externe.
        liaison_cols: Colonnes identifiant une liaison.

    Returns:
        Série indexée par liaison, part (0-1) des retards d'origine interne.
    """
    data_ok = data[data["cause_fiable"]]
    colonnes_nb_retard = {f"nb_retard_{famille}": famille for famille in familles}
    agg = data_ok.groupby(liaison_cols)[list(colonnes_nb_retard)].sum()

    internes = [colonne for colonne, famille in colonnes_nb_retard.items() if famille in ("reseau", "voyageurs")]
    total = agg.sum(axis=1)
    return agg[internes].sum(axis=1) / total


### Test et validation des fonctions Phase 4

Même principe qu'en Phase 3 : reconstruire depuis zéro avec les fonctions et comparer au calcul
manuel (`par_liaison_12m`).


In [17]:
df_test2 = load(REGULARITE)
for col in liaison:
    df_test2[col] = normaliser_noms(df_test2[col])

# Classe de durée (Phase 2)
duree_test = (
    df_test2.groupby(liaison)["duree_moyenne"].median().rename("duree_liaison").reset_index()
)
duree_test["classe_duree"] = pd.cut(
    duree_test["duree_liaison"], bins=[0, 90, 180, float("inf")], labels=["< 1h30", "1h30-3h", "> 3h"]
)
df_test2 = df_test2.merge(duree_test, on=liaison)

df_test2["cause_fiable"] = marquer_cause_fiable(df_test2)
df_test2 = ajouter_nb_retard_par_famille(df_test2)

recent_test = agreger_derniers_mois(df_test2, n_mois=12)
comp_test = composantes_fiabilite(recent_test)
score_test = score_fiabilite(comp_test).reindex(par_liaison_12m.index)
interne_test = part_origine_interne(recent_test).reindex(par_liaison_12m.index)

pd.testing.assert_frame_equal(
    score_test[COMPOSANTES_FIABILITE], par_liaison_12m[COMPOSANTES_FIABILITE], check_exact=False
)
pd.testing.assert_series_equal(
    score_test["score_fiabilite"],
    par_liaison_12m["score_fiabilite"],
    check_exact=False,
    check_names=False,
)
pd.testing.assert_series_equal(
    interne_test, par_liaison_12m["part_origine_interne"], check_exact=False, check_names=False
)
print("OK : les fonctions Phase 4 reproduisent exactement le calcul manuel.")


OK : les fonctions Phase 4 reproduisent exactement le calcul manuel.


## Résumé et prochaines étapes

**Phase 3 — Saisonnalité des causes** (fonctions validées : `marquer_cause_fiable`,
`ajouter_nb_retard_par_famille`, `contributions_par_saison`)
- Le Réseau domine en hiver, printemps et automne ; les Voyageurs passent devant en été
  (contribution agrégée), à nuancer : le Réseau reste majoritaire année par année même l'été.
- L'Externe reste stable et minoritaire sur toutes les saisons.
- À documenter dans la datasheet : la nuance agrégé vs par année.

**Phase 4 — Score de fiabilité V1** (fonctions validées : `agreger_derniers_mois`,
`composantes_fiabilite`, `score_fiabilite`, `part_origine_interne`)
- 121 liaisons scorées sur les 12 derniers mois (oct. 2025 → sept. 2026), 3 composantes à poids
  égaux, rang par classe de durée.
- Les liaisons les moins fiables recoupent les constats de la Phase 2 (Paris Est ↔
  Stuttgart/Francfort, Douai/Arras → Paris Nord).
- Part d'origine interne médiane : 73 %, affichée à côté du score sans l'influencer.
- **Limite à trancher avec le groupe** : seuil de retard par classe de durée (5/10/15 min) non
  disponible dans les données → proxy `nb_train_retard_arrivee` utilisé pour toutes les classes.

### À faire ensuite
1. Copier les fonctions validées dans `railinsight/analysis/causes.py` (Phase 3) et
   `railinsight/analysis/reliability.py` (Phase 4), avec les docstrings déjà rédigées ici.
2. Écrire les tests unitaires correspondants dans `test/` (ex. `test_causes.py`, `test_reliability.py`),
   en réutilisant `test/fixtures/mini_sncf.csv` comme pour les autres modules.
3. Brancher `app/pages/03_saisons.py` sur `contributions_par_saison` + `cause_stacked_bar`
   (déjà disponible dans `railinsight/viz/charts.py`).
4. Brancher `app/pages/04_ma_liaison.py` sur `score_fiabilite` + `part_origine_interne`, avec un
   sélecteur de liaison.
5. Vérifier `ruff check .` et `pytest` (coverage ≥ 90 %) avant d'ouvrir la PR vers `develop`.
